# 🚀 VisionForge - Master End-to-End Training & Multi-Model Forensic Benchmark
**Author:** Sushant Gabhane (MSc IT Project - Roll/ID: D09)

**Architecture:** Large YOLO Detection Backbone (70 Epochs | AdamW | Cosine LR)

**Dataset:** `yolo_raw_all_frames_keyframe.zip` (2,391 Raw Frames with Mathematical Rigid Dimension Lock)

**Multi-Model Audit:** Benchmarking `best (8).pt`, `best (9).pt`, `best.pt`, and `best_keyframe_70ep.pt`

**Video Verification:** Side-by-Side Annotation Comparison on `sushant_gabhane_D09_20261009102400.mp4`

---
### 📋 Pipeline Workflow
1. **GPU & Drive Discovery:** Auto-mounts Google Drive and detects the `models` / `models\` folder.
2. **NVMe SSD Dataset Extraction:** Unzips 2,391 raw images to local SSD for 10x-15x faster epoch speed.
3. **Industrial Training (70 Epochs):** Trains YOLO-Large with Cosine Annealing, AdamW, and CIoU loss gain.
4. **Multi-Model Forensic Audit:** Runs standard validation across ALL `.pt` files and generates a comparative scorecard.
5. **Side-by-Side Video Generator:** Renders a high-clarity H.264 video comparing previous model vs. new keyframe model.
6. **Drive Archival:** Automatically copies weights, reports, charts, and video to `models/keyframe_benchmark_results/`.

In [ ]:
#@title Step 1: Mount Google Drive & Verify GPU Acceleration
import os, torch

# Check if Drive is already mounted
if os.path.exists('/content/drive/MyDrive'):
    print('✅ Google Drive is ALREADY mounted and accessible at /content/drive/MyDrive')
else:
    print('Connecting to Google Drive...')
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        print('✅ Google Drive mounted successfully!')
    except Exception as err:
        print(f'⚠️ Notice: {err}\nPlease click the folder icon on the left sidebar, click Mount Drive, and reconnect.')

print('=' * 80)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'✅ GPU Acceleration Active: {gpu_name} ({vram:.2f} GB VRAM)')
else:
    print('⚠️ WARNING: No GPU detected! Go to Runtime > Change runtime type > Select T4 GPU')
print('=' * 80)


In [ ]:
#@title Step 2: Install Packages & Prepare Dataset on Fast NVMe SSD
!pip install -q -U ultralytics pyyaml opencv-python tabulate

import os, sys, shutil, zipfile, yaml

def find_drive_folder():
    drive_root = '/content/drive/MyDrive'
    candidates = [os.path.join(drive_root, 'models'), os.path.join(drive_root, 'models\\'), drive_root]
    for c in candidates:
        if os.path.exists(c) and os.path.isdir(c):
            return c
    if os.path.exists(drive_root):
        for entry in os.listdir(drive_root):
            if entry.lower().startswith('models'):
                p = os.path.join(drive_root, entry)
                if os.path.isdir(p):
                    return p
    return drive_root

def find_file(filename, priority_dir=None):
    search_dirs = []
    if priority_dir and os.path.exists(priority_dir):
        search_dirs.append(priority_dir)
    drive_root = '/content/drive/MyDrive'
    search_dirs.extend([os.path.join(drive_root, 'models'), os.path.join(drive_root, 'models\\'), drive_root, '/content'])
    for d in search_dirs:
        if os.path.exists(d):
            direct = os.path.join(d, filename)
            if os.path.exists(direct):
                return direct
            for entry in os.listdir(d):
                if entry.strip().lower() == filename.strip().lower():
                    return os.path.join(d, entry)
    if os.path.exists(drive_root):
        for root, _, files in os.walk(drive_root):
            for f in files:
                if f.strip().lower() == filename.strip().lower():
                    return os.path.join(root, f)
    return None

models_folder = find_drive_folder()
print(f'📁 Active Google Drive Folder: {models_folder}')

zip_source = find_file('yolo_raw_all_frames_keyframe.zip', priority_dir=models_folder)
if not zip_source:
    raise FileNotFoundError('❌ Could not locate yolo_raw_all_frames_keyframe.zip! Check Google Drive upload status.')

print(f'📦 Found dataset archive at: {zip_source} ({os.path.getsize(zip_source)/(1024**2):.1f} MB)')
local_dataset_dir = '/content/dataset_keyframe'
if os.path.exists(local_dataset_dir):
    shutil.rmtree(local_dataset_dir)

print('Extracting 2,391 raw images & labels to local SSD (/content/dataset_keyframe)...')
with zipfile.ZipFile(zip_source, 'r') as zf:
    zf.extractall(local_dataset_dir)

yaml_path = os.path.join(local_dataset_dir, 'data.yaml')
with open(yaml_path, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['path'] = local_dataset_dir
cfg['train'] = 'images/train'
cfg['val'] = 'images/val'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('✅ data.yaml configured for high-speed local NVMe training:')
print(open(yaml_path).read().strip())


In [ ]:
#@title Step 3: Launch 70-Epoch Training on Keyframe Ground Truth
from ultralytics import YOLO

base_weights = 'yolov8l.pt'
print(f'🎯 Architecture: YOLO-Large ({base_weights}) | Target: 70 Epochs')
print('⚙️ Optimal Industrial Hyperparameters: AdamW | Cosine LR | CIoU Box Gain: 7.5 | Close Mosaic: 10')

model_new = YOLO(base_weights)

train_results = model_new.train(
    data=yaml_path,
    epochs=70,
    imgsz=640,
    batch=8,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3.0,
    close_mosaic=10,
    box=7.5,
    cls=0.5,
    dfl=1.5,
    patience=25,
    save=True,
    save_period=10,
    amp=True,
    workers=4,
    project='/content/runs/detect',
    name='keyframe_large_70epochs',
    exist_ok=True,
    plots=True
)

new_best_weights = '/content/runs/detect/keyframe_large_70epochs/weights/best.pt'
print(f'\n✅ Training complete! New best weights generated at: {new_best_weights}')


In [ ]:
#@title Step 4: Multi-Model Forensic Audit (Evaluating ALL .pt Files)
import time, json
import numpy as np

discovered_models = []
seen_paths = set()

# 1. Newly trained 70-epoch model
if os.path.exists(new_best_weights):
    discovered_models.append({
        'name': 'New Keyframe Model (70 Epochs)',
        'path': new_best_weights,
        'notes': 'Trained on 2,391 raw keyframe frames (Rigid Lock Ground Truth)'
    })
    seen_paths.add(os.path.abspath(new_best_weights))

# 2. All models in Google Drive models folder
if os.path.exists(models_folder):
    for f in sorted(os.listdir(models_folder)):
        if f.endswith('.pt') and 'keyframe' not in f:
            full_p = os.path.join(models_folder, f)
            if os.path.abspath(full_p) not in seen_paths:
                seen_paths.add(os.path.abspath(full_p))
                discovered_models.append({
                    'name': f'Drive Checkpoint: {f}',
                    'path': full_p,
                    'notes': 'Historical checkpoint in Drive models folder'
                })

audit_results = []
val_images_dir = os.path.join(local_dataset_dir, 'images', 'val')
sample_imgs = [os.path.join(val_images_dir, f) for f in os.listdir(val_images_dir)[:15]]

for m_info in discovered_models:
    m_path = m_info['path']
    m_name = m_info['name']
    if not os.path.exists(m_path): continue
    
    fsize_mb = os.path.getsize(m_path) / (1024 * 1024)
    print(f'\nEvaluating: {m_name} ({fsize_mb:.2f} MB)...')
    try:
        eval_model = YOLO(m_path)
        params_count = sum(p.numel() for p in eval_model.model.parameters())
        val_res = eval_model.val(data=yaml_path, split='val', imgsz=640, batch=8, verbose=False)
        
        map50 = float(val_res.box.map50) * 100
        map50_95 = float(val_res.box.map) * 100
        precision = float(val_res.box.mp) * 100
        recall = float(val_res.box.mr) * 100
        
        latencies = []
        for simg in sample_imgs:
            t_inf_start = time.perf_counter()
            _ = eval_model(simg, verbose=False)
            latencies.append((time.perf_counter() - t_inf_start) * 1000)
        avg_latency = float(np.median(latencies))
        
        audit_results.append({
            'model_name': m_name,
            'file_name': os.path.basename(m_path),
            'file_size_mb': round(fsize_mb, 2),
            'total_parameters': params_count,
            'precision_mp': round(precision, 2),
            'recall_mr': round(recall, 2),
            'map_50': round(map50, 2),
            'map_50_95': round(map50_95, 2),
            'latency_ms': round(avg_latency, 2),
            'notes': m_info['notes']
        })
        print(f'  ✓ mAP@50: {map50:.2f}% | mAP@50-95: {map50_95:.2f}% | Latency: {avg_latency:.1f}ms')
    except Exception as e:
        print(f'  ❌ Error evaluating {m_name}: {e}')

print('\n' + '=' * 95)
print('                   MASTER FORENSIC AUDIT: ALL MODEL CHECKPOINTS')
print('=' * 95)
print(f"{'Model Name':<34} | {'Size (MB)':<10} | {'Params (M)':<11} | {'mAP@50':<9} | {'mAP@50-95':<11} | {'Latency':<8}")
print('-' * 95)
for res in audit_results:
    p_millions = f"{res['total_parameters'] / 1e6:.2f}M"
    print(f"{res['model_name']:<34} | {res['file_size_mb']:<10.1f} | {p_millions:<11} | {res['map_50']:<8.2f}% | {res['map_50_95']:<10.2f}% | {res['latency_ms']:<6.1f}ms")
print('=' * 95)

report_json_path = '/content/multi_model_forensic_report.json'
with open(report_json_path, 'w') as jf:
    json.dump(audit_results, jf, indent=2)

report_md_path = '/content/multi_model_forensic_report.md'
with open(report_md_path, 'w') as mf:
    mf.write('# VisionForge Multi-Model Forensic Benchmark Report\n\n')
    mf.write('| Model Name | File Size (MB) | Parameters | mAP@50 (%) | mAP@50-95 (%) | Latency (ms) |\n')
    mf.write('| :--- | :--- | :--- | :--- | :--- | :--- |\n')
    for r in audit_results:
        mf.write(f"| **{r['model_name']}** | {r['file_size_mb']} MB | {r['total_parameters']/1e6:.2f}M | {r['map_50']}% | {r['map_50_95']}% | {r['latency_ms']} ms |\n")


In [ ]:
#@title Step 5: Side-by-Side Video Annotation Comparison Engine
import cv2, subprocess

video_target_name = 'sushant_gabhane_D09_20261009102400.mp4'
video_path = find_file(video_target_name, priority_dir=models_folder)
if not video_path and os.path.exists(models_folder):
    for f in os.listdir(models_folder):
        if f.endswith('.mp4'):
            video_path = os.path.join(models_folder, f)
            break

prev_model_candidates = ['best (9).pt', 'best (8).pt', 'best.pt']
prev_model_path = None
for cand in prev_model_candidates:
    p = find_file(cand, priority_dir=models_folder)
    if p and os.path.exists(p):
        prev_model_path = p
        break

out_video_h264 = '/content/side_by_side_model_comparison.mp4'
preview_snapshots = []

if video_path and os.path.exists(video_path) and prev_model_path and os.path.exists(prev_model_path):
    print(f'🎬 Processing video: {video_path}')
    print(f'   Model A (Left - Red):    Previous Checkpoint ({os.path.basename(prev_model_path)})')
    print(f'   Model B (Right - Green):  New 70-Epoch Keyframe Model ({os.path.basename(new_best_weights)})')
    
    model_prev = YOLO(prev_model_path)
    model_curr = YOLO(new_best_weights)
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    start_f = min(750, max(0, total_frames - 550))
    cap.set(cv2.CAP_PROP_POS_FRAMES, start_f)
    
    vw, vh = 640, 360
    out_w, out_h = vw * 2, vh + 60
    temp_raw_vid = '/content/temp_comparison.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(temp_raw_vid, fourcc, 15.0, (out_w, out_h))
    
    frames_to_process = min(400, total_frames - start_f)
    print(f'Rendering {frames_to_process} side-by-side frames...')
    snapshot_indices = [int(frames_to_process * 0.15), int(frames_to_process * 0.50), int(frames_to_process * 0.85)]
    
    for i in range(frames_to_process):
        ret, frame = cap.read()
        if not ret: break
        cur_frame_num = start_f + i
        
        res_a = model_prev(frame, conf=0.25, verbose=False)[0]
        res_b = model_curr(frame, conf=0.25, verbose=False)[0]
        
        panel_a = cv2.resize(frame, (vw, vh))
        panel_b = cv2.resize(frame, (vw, vh))
        
        conf_a_str = 'None'
        box_a_dims = 'N/A'
        for box in res_a.boxes:
            bx1, by1, bx2, by2 = map(int, box.xyxy[0].tolist())
            sx1 = int(bx1 * vw / frame.shape[1]); sy1 = int(by1 * vh / frame.shape[0])
            sx2 = int(bx2 * vw / frame.shape[1]); sy2 = int(by2 * vh / frame.shape[0])
            conf_val = float(box.conf[0])
            conf_a_str = f'{conf_val * 100:.1f}%'
            box_a_dims = f'{bx2-bx1}x{by2-by1}'
            cv2.rectangle(panel_a, (sx1, sy1), (sx2, sy2), (50, 70, 240), 2)
            cv2.putText(panel_a, f'Package {conf_a_str}', (sx1, max(18, sy1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (80, 110, 255), 2)
            
        cv2.rectangle(panel_a, (0, 0), (vw, 36), (20, 20, 30), -1)
        cv2.putText(panel_a, f'PREVIOUS MODEL: {os.path.basename(prev_model_path)}', (12, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (80, 110, 255), 2)
        
        conf_b_str = 'None'
        box_b_dims = 'N/A'
        for box in res_b.boxes:
            bx1, by1, bx2, by2 = map(int, box.xyxy[0].tolist())
            sx1 = int(bx1 * vw / frame.shape[1]); sy1 = int(by1 * vh / frame.shape[0])
            sx2 = int(bx2 * vw / frame.shape[1]); sy2 = int(by2 * vh / frame.shape[0])
            conf_val = float(box.conf[0])
            conf_b_str = f'{conf_val * 100:.1f}%'
            box_b_dims = f'{bx2-bx1}x{by2-by1}'
            cv2.rectangle(panel_b, (sx1, sy1), (sx2, sy2), (30, 210, 80), 2)
            cv2.putText(panel_b, f'Package {conf_b_str}', (sx1, max(18, sy1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (100, 240, 120), 2)
            
        cv2.rectangle(panel_b, (0, 0), (vw, 36), (20, 30, 20), -1)
        cv2.putText(panel_b, 'NEW 70-EPOCH KEYFRAME MODEL', (12, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (100, 240, 120), 2)
        
        frame_body = np.hstack([panel_a, panel_b])
        bar = np.full((60, out_w, 3), 24, dtype=np.uint8)
        telemetry_txt = f'Frame {cur_frame_num}/{total_frames} | Previous: Conf {conf_a_str} (Box {box_a_dims}) VS Keyframe: Conf {conf_b_str} (Box {box_b_dims} Rigid Lock)'
        cv2.putText(bar, telemetry_txt, (20, 38), cv2.FONT_HERSHEY_SIMPLEX, 0.48, (220, 220, 220), 1)
        
        composed = np.vstack([frame_body, bar])
        writer.write(composed)
        
        if i in snapshot_indices:
            snap_path = f'/content/side_by_side_preview_frame_{cur_frame_num}.png'
            cv2.imwrite(snap_path, composed)
            preview_snapshots.append(snap_path)
            
        if (i + 1) % 100 == 0 or (i + 1) == frames_to_process:
            print(f'  ✓ Rendered frame {i + 1} / {frames_to_process} ({(i + 1) / frames_to_process * 100:.1f}%)')
            
    writer.release()
    cap.release()
    
    print('\nRe-encoding video to browser-compatible H.264 format...')
    subprocess.run(['ffmpeg', '-y', '-i', temp_raw_vid, '-c:v', 'libx264', '-pix_fmt', 'yuv420p', '-preset', 'fast', out_video_h264], check=True)
    if os.path.exists(temp_raw_vid): os.remove(temp_raw_vid)
    print(f'✅ Comparison video generated: {out_video_h264} ({os.path.getsize(out_video_h264)/(1024**2):.1f} MB)')
else:
    print('⚠️ Notice: Video or previous model not found. Skipping video render.')


In [ ]:
#@title Step 6: Automatically Save All Deliverables to Google Drive
drive_backup_dir = os.path.join(models_folder, 'keyframe_benchmark_results')
os.makedirs(drive_backup_dir, exist_ok=True)
runs_output_dir = '/content/runs/detect/keyframe_large_70epochs'

artifacts = [
    (new_best_weights, 'best_keyframe_70ep.pt'),
    (os.path.join(runs_output_dir, 'weights/last.pt'), 'last_keyframe_70ep.pt'),
    (os.path.join(runs_output_dir, 'results.csv'), 'training_results.csv'),
    (os.path.join(runs_output_dir, 'results.png'), 'training_loss_curves.png'),
    (os.path.join(runs_output_dir, 'confusion_matrix.png'), 'confusion_matrix.png'),
    (os.path.join(runs_output_dir, 'F1_curve.png'), 'F1_curve.png'),
    (report_json_path, 'multi_model_forensic_report.json'),
    (report_md_path, 'multi_model_forensic_report.md'),
]
if os.path.exists(out_video_h264):
    artifacts.append((out_video_h264, 'side_by_side_model_comparison.mp4'))
for snap in preview_snapshots:
    artifacts.append((snap, os.path.basename(snap)))

print(f'💾 Securing outputs directly to Google Drive:\n   -> {drive_backup_dir}')
for src, out_name in artifacts:
    if os.path.exists(src):
        dst = os.path.join(drive_backup_dir, out_name)
        shutil.copy2(src, dst)
        print(f'  ✓ Saved: {out_name:<34} ({os.path.getsize(dst)/(1024**2):.2f} MB)')

print('\n🎉 ALL DELIVERABLES PERMANENTLY SAVED TO GOOGLE DRIVE!')


In [ ]:
#@title Step 7: View Loss Curves & Side-by-Side Comparison Inline
from IPython.display import Image, display

loss_curve = os.path.join(runs_output_dir, 'results.png')
if os.path.exists(loss_curve):
    print('\n--- Training Loss & Metric Curves ---')
    display(Image(filename=loss_curve))

if preview_snapshots:
    print('\n--- Side-by-Side Model Comparison Snapshots ---')
    for snap in preview_snapshots:
        print(f'Snapshot: {os.path.basename(snap)}')
        display(Image(filename=snap))
